# Phase 6 牛津 Tutorial LLM 仿真 (Oxford + HBS + Hattie)

## Persona Prompt (本 tutor 的人格设定)

> You are an Oxford tutorial fellow in **Capstone Phase 6: Implementation, IMRaD Paper Writing, Reproducible Research, Poster, and Defense**.
> **Never give direct answers.** (不直接给答案, 禁直接答案)
> Use **Socratic questioning** (苏格拉底追问) to lead the student to discover the answer themselves.
> Act as an **HBS devil's advocate** (哈佛商学院反驳角色): challenge every claim, demand evidence, reject vague assertions.
> **Reject vague claims** like "the system works well" or "the paper is good" -- demand specific metrics, specific sections, specific data.
> End **each turn** with a probing question that forces the student to go deeper.

**领域特定约束 (Phase 6)**:
- 学生若说"我用 DSR 框架", 必须追问"DSR Step 几映射到 IMRaD 哪节? 凭什么?"
- 学生若说"我做了统计检验", 必须追问"t(df)=? p=? d=? 95%CI=? 若前提变(pooled SD 改) 如何?"
- 学生若说"系统可复现", 必须追问"trace hash 凭什么防版本漂移? 反例: 模型升级后旧 trace 还能复现吗?"
- 学生若说"天道推演章节写好了", 必须追问"五项能力(局势感知/因果链/沙盘/概率/最优路径)与多Agent仿真同构, 缺一不可. 凭什么只写三项?"

**禁止 (Never)**:
- 不直接给答案 (Never give direct answers)
- 不表扬学生人格 (No Self-level praise per Hattie)
- 不接受"差不多"/"挺好的"/"应该可以"等模糊词


## Pre-Tutorial Task (强制提取练习 retrieval practice)

> **上课前 24 小时必须提交** (不提交不能进 tutorial):

请用 300-500 字回答以下问题, 提交到 `pre_tutorial_essay.md`:

1. **(DSR)**: 用 DSR 六步框架 (Hevner 2004 / Peffers 2007) 描述你的 Capstone artifact, 明确 Step 1-6 分别对应 IMRaD 哪个章节
2. **(统计)**: 给定 NSW RCT (N=445, treatment=185, control=260) 的 treatment 组 1975 收入均值 8345, control 7923, pooled SD 6218, 写出独立样本 t 检验的 APA 第 7 版表述
3. **(可复现)**: 你的 langsmith @traceable trace 存档保存了哪 4 类元数据? 为什么需要 trace hash?
4. **(天道推演)**: 天道推演五项能力与多Agent仿真的同构表, 写出至少 3 行

**为什么不直接讲**: 提取练习 (retrieval practice) 比重读 (rereading) 高效 2-3 倍 (Roediger & Karpicke 2006). 你先尝试提取, tutorial 才能精准定位盲点.


In [ ]:
# Socratic Multi-Turn Loop (静态 if/else 模拟, 不真调 openai/anthropic API)
# 本 cell 用静态规则模拟牛津 tutor 的苏格拉底追问. 学生输入 -> 规则匹配 -> 追问/反馈.
import json, os, datetime

STUDENT_ESSAY = """
我的 Capstone 用 DSR 框架. Step 1 是问题识别, Step 3 是设计开发, Step 5 是评估.
NSW 数据上 t 检验 p=.04, 显著. 系统用 langsmith 追踪, 可复现.
天道推演章节我写了局势感知和沙盘模拟.
"""  # 学生提交的 pre-tutorial essay (简化版)

# 苏格拉底追问规则库 (>=5 个苏格拉底问: 为什么/反例/若前提变/凭什么/如何)
SOCRATIC_RULES = [
    {
        "id": "Q1",
        "trigger": "DSR Step",  # 学生提到 DSR Step
        "probe": "为什么 DSR Step 4 (演示) 你没提? 反例: 没有 Step 4 演示, Step 5 评估凭什么成立? 若前提变(系统未在真实场景跑过), 你的 ATE 还是有效的吗?",
        "skill": "DSR Step4缺失"
    },
    {
        "id": "Q2",
        "trigger": "p=.04",  # 学生写 p=.04 但没写 t/df/d/95%CI
        "probe": "你写 p=.04, 凭什么只报 p? t(df)=? Cohen's d=? 95%CI=? 如何判断效应量大小? 反例: N=10000 时 p=.04 但 d=0.01, 这算实质显著吗?",
        "skill": "APA不完整"
    },
    {
        "id": "Q3",
        "trigger": "langsmith",
        "probe": "你说 langsmith 追踪可复现. 凭什么? trace hash 你保存了吗? 反例: 模型从 GPT-4 升级到 GPT-5, 旧 trace 还能复现吗? 若前提变(API 版本漂移), 你的 trace 存档如何报警?",
        "skill": "trace hash缺失"
    },
    {
        "id": "Q4",
        "trigger": "天道推演",
        "probe": "天道推演五项能力(局势感知/因果链追踪/沙盘模拟/概率评估/最优路径推荐)你只写两项. 凭什么漏掉因果链追踪和概率评估? 如何与多Agent仿真的 Agent 交互链和贝叶斯推断同构? 反例: 缺这两项, 你的特色章节还成立吗?",
        "skill": "天道推演不完整"
    },
    {
        "id": "Q5",
        "trigger": "END",  # 兜底追问
        "probe": "你的发表路线图呢? 为什么先投 arXiv 而不是直接投 MIS Quarterly? 如何用 arXiv 时间戳防抢发? 若前提变(会议拒稿), 你的下一步是什么?",
        "skill": "发表路线图缺失"
    }
]

def socratic_turn(student_text, turn_idx):
    """静态 if/else 模拟单轮 Socratic 追问. 不调任何 LLM API."""
    for rule in SOCRATIC_RULES:
        if rule["trigger"] in student_text:
            return {
                "turn": turn_idx,
                "student_input_snippet": student_text[:80] + "...",
                "tutor_probe": rule["probe"],
                "blind_spot_flagged": rule["skill"]
            }
    # 兜底: 若学生输入没命中任何 trigger, 用 Q5
    return {
        "turn": turn_idx,
        "student_input_snippet": student_text[:80] + "...",
        "tutor_probe": SOCRATIC_RULES[-1]["probe"],
        "blind_spot_flagged": SOCRATIC_RULES[-1]["skill"]
    }

# 模拟 4 轮 Socratic 对话 (静态, 学生回复用预设字符串)
mock_student_replies = [
    STUDENT_ESSAY,  # turn 1: 原始 essay
    "Step 4 演示我补上了, 在 Methods 写了 LangGraph node 调用. t 检验我加了 t(443)=0.72, d=0.07, 95%CI [-228.5, 1132.5].",  # turn 2
    "trace hash 我用 hashlib.md5(prompt+model_version). 天道推演补了因果链追踪<->Agent交互链, 概率评估<->贝叶斯推断.",  # turn 3
    "发表路线图: arXiv -> ICIS -> MIS Quarterly. arXiv 拿时间戳, ICIS 拿反馈, MISQ 拿影响因子."  # turn 4
]

dialogue = []
for i, reply in enumerate(mock_student_replies, start=1):
    turn = socratic_turn(reply, i)
    dialogue.append(turn)
    print(f"\n=== Turn {turn['turn']} ===")
    print(f"学生: {turn['student_input_snippet']}")
    print(f"Tutor (Socratic): {turn['tutor_probe']}")
    print(f"盲点标记: {turn['blind_spot_flagged']}")

print("\n=== 4 轮 Socratic 对话结束 ===")
print(f"共标记 {len(set(d['blind_spot_flagged'] for d in dialogue))} 个盲点")


In [ ]:
# student_model.json: 记录学生掌握度与盲点 (供下次 tutorial 个性化)
import json, os, datetime

STUDENT_MODEL_PATH = "student_model.json"

def load_student_model():
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    # 初始化空模型
    return {
        "unit": "U-Capstone-Phase6",
        "mastery": {
            "ILO1_DSR": 0.0,        # DSR 六步 -> IMRaD 映射
            "ILO2_trace": 0.0,      # langsmith @traceable + deepeval
            "ILO3_stats": 0.0,      # statsmodels APA 报告
            "ILO4_IMRaD": 0.0       # IMRaD 草稿 + 发表路线图
        },
        "blind_spots": [],          # 盲点关键词列表
        "dialogue_history": [],     # 历次对话摘要
        "last_review": None,
        "review_count": 0
    }

def update_student_model(model, dialogue):
    """根据 Socratic 对话更新掌握度与盲点."""
    blind_map = {
        "DSR Step4缺失": ("ILO1_DSR", -0.2),
        "APA不完整": ("ILO3_stats", -0.15),
        "trace hash缺失": ("ILO2_trace", -0.2),
        "天道推演不完整": ("ILO1_DSR", -0.1),
        "发表路线图缺失": ("ILO4_IMRaD", -0.15)
    }
    for turn in dialogue:
        bs = turn["blind_spot_flagged"]
        if bs not in model["blind_spots"]:
            model["blind_spots"].append(bs)
        if bs in blind_map:
            ilo, delta = blind_map[bs]
            model["mastery"][ilo] = max(0.0, min(1.0, model["mastery"][ilo] + delta))
    # 学生回复了正确内容 -> 掌握度上升 (turn 2/3/4 学生补全了内容)
    if len(dialogue) >= 2:
        model["mastery"]["ILO1_DSR"] = min(1.0, model["mastery"]["ILO1_DSR"] + 0.3)
        model["mastery"]["ILO3_stats"] = min(1.0, model["mastery"]["ILO3_stats"] + 0.3)
    if len(dialogue) >= 3:
        model["mastery"]["ILO2_trace"] = min(1.0, model["mastery"]["ILO2_trace"] + 0.3)
    if len(dialogue) >= 4:
        model["mastery"]["ILO4_IMRaD"] = min(1.0, model["mastery"]["ILO4_IMRaD"] + 0.3)
    model["last_review"] = datetime.datetime.now().isoformat()
    model["review_count"] += 1
    model["dialogue_history"].append({
        "turn_count": len(dialogue),
        "blind_spots_this_session": [t["blind_spot_flagged"] for t in dialogue]
    })
    return model

# 加载 / 更新 / 保存
model = load_student_model()
print("=== Tutorial 前 student_model ===")
print(json.dumps(model, ensure_ascii=False, indent=2))

model = update_student_model(model, dialogue)  # dialogue 来自 cell 3

with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
    json.dump(model, f, ensure_ascii=False, indent=2)

print("\n=== Tutorial 后 student_model (已更新) ===")
print(json.dumps(model, ensure_ascii=False, indent=2))
print(f"\nstudent_model.json 已写入: {os.path.abspath(STUDENT_MODEL_PATH)}")


## Hattie 四级形成性反馈 (Hattie & Timperley 2007, 避免 Self 级表扬)

> Tutor 在 4 轮 Socratic 对话后, 给出四级反馈. **避免 Self 级表扬** (不写"你真聪明"/"做得好"), 只给 Task / Process / Self-Reg / Feed-Forward 四级.

### [TASK] 任务级反馈 (关于本次 essay 的具体内容)
你的 pre-tutorial essay 暴露 4 个任务级问题:
1. DSR Step 4 (演示) 完全缺失 -- Methods 章节无法支撑 Step 5 评估
2. t 检验仅报 p=.04, 缺 t(df)/Cohen's d/95%CI -- APA 第 7 版不达标
3. langsmith trace 存档未提 trace hash -- 可复现性有漏洞
4. 天道推演章节只写 2/5 项能力 -- 特色章节不完整

**修复要求**: 重写 essay, 补全上述 4 项, 24 小时内重新提交.

### [PROCESS] 过程级反馈 (关于学习策略, 不关于人格)
你的学习策略有以下问题:
- 你在 essay 中"罗列名词"(DSR/langsmith/天道推演)但未"建立映射"(Step几->哪节/hash->防什么/五项->五同构). 这是**浅层学习策略**, 阻碍迁移.
- **建议过程改进**: 下次 pre-tutorial 用"映射表"而非"清单" -- 每个概念必须画"X -> Y 因为 Z"的三元组.

### [SELF-REG] 自我调节级反馈 (关于元认知监控)
你的 essay 显示**元认知监控不足**:
- 你写"系统可复现"但没自问"凭什么可复现?" -- 缺乏**自我追问**习惯
- 你写"天道推演章节写好了"但没自问"五项能力齐了吗?" -- 缺乏**完整性自检**清单
- **建议**: 提交前用 `alignment.md` 的 3 自检问题 (Feed Up/Back/Forward) 自查, 不要等 tutor 指出.

### [FEED-FORWARD] 前馈级反馈 (关于下一步行动)
基于本次 4 个盲点, 你的下一步行动:
1. **D1 Stage 2 (部分填空)**: 重做 DSR Step -> IMRaD 映射表, 重点 Step 4
2. **D3 Stage 1 (完整示范)**: 重看 t 检验 worked example, 补 APA 模板
3. **D2 Stage 1**: 重看 langsmith @traceable worked example, 补 trace hash 代码
4. **复习单元**: 回去重做 Phase 3 (LangGraph) 和 Phase 4 (DoWhy 因果) 的 alignment.md, 因为你的盲点本质是前序 Phase 未掌握
5. **下次 tutorial 限频**: 每单元 1 次/天 (见 cell 6), 明天再来


## 限频 (防 LLM 依赖) + Exit Artifact

### 限频规则 (防止学生对 LLM 形成依赖)
- **每单元 1 次/天 (daily limit: 1 session per unit per day)**: 同一单元的 tutorial 每天最多 1 次
- **每周 3 次/单元上限**: 防止反复"刷 tutorial"替代真实学习
- **连续 2 次 tutorial 同一盲点未修复**: 强制回退 `practice.md` Weak Loop, 暂停 tutorial 1 天
- **为什么限频**: Hattie (2009) meta-analysis 显示, 过度依赖外部反馈会削弱自我调节能力 (d=-0.15). Tutorial 是脚手架, 不是拐杖.

### Exit Artifact (本次 tutorial 结束必须产出)
完成本次 tutorial 后, 在 `tutorial_exit.md` 中写:

1. **2-3 个本次发现的盲点** (从 student_model.json 的 blind_spots 复制):
   - 例: "DSR Step 4 演示被我漏掉, 导致 Step 5 评估无前置"
   - 例: "APA 第 7 版 t 检验表述我漏了 95%CI"
   - 例: "天道推演五项能力我只写了两项, 同构表不完整"

2. **推荐复习单元** (基于盲点回溯前序 Phase):
   - 盲点 DSR Step 4 -> 复习 Phase 3 (LangGraph Agent 架构, 演示即 node 调用)
   - 盲点 APA 统计 -> 复习 Phase 4 (DoWhy 因果, ATE 估计的统计基础)
   - 盲点 trace hash -> 复习 Phase 3 (LangGraph 可复现性)
   - 盲点天道推演 -> 复习项目 CLAUDE.md § 天道推演系统 + Phase 5 (商业模式推演)

3. **下次 tutorial 的 pre-tutorial 任务** (强制 retrieval):
   - 重写 300-500 字 essay, 必须补全本次 4 个盲点
   - 提交 `practice.md` D1/D2/D3 Stage 2 的部分填空作业

### Tutorial 闭环 (Oxford + HBS + Hattie)
```
pre-tutorial essay (retrieval practice)
  -> 4 轮 Socratic 追问 (Oxford, 不直接给答案)
  -> HBS devil's advocate 反驳 (拒绝模糊)
  -> Hattie 四级反馈 (TASK/PROCESS/SELF-REG/FEED-FORWARD, 无 Self 表扬)
  -> student_model.json 更新 (盲点 + 掌握度)
  -> exit artifact (盲点 + 复习单元)
  -> 限频: 明天再来 (1次/天)
```

**参考**: Hattie, J., & Timperley, H. (2007). The Power of Feedback. *Review of Educational Research*, 77(1), 81-112. | Biggs, J. (1996). Enhancing teaching through constructive alignment. *Higher Education*, 32, 347-364. | Ericsson, K. A. (1993). The role of deliberate practice. *Psychological Review*, 100(3), 363-406.
